# GPBSF PointNet++ (SSG) Google Colab 训练 Notebook (Seed 3409)

- **云盘联动**：直接从绑定的 Google Drive 根目录读取代码包与数据集，输出目录 `runs/` 实时软链接至 Google Drive。
- **断线永不丢失**：即使 Colab 5 小时超时或断开连接释放虚拟机，所有断点 `last.pt` 与日志均实时保存在云盘，重连后可随时无缝接着跑。
- **单 GPU 极速架构**：全量 RAM 预载（零磁盘 I/O）+ TorchScript JIT 编译 FPS/BallQuery 向量化算子。
- **目标任务**：PointNet++ SSG 模型，随机种子 `seed=3409`，训练轮次 `epochs=125`。

In [ ]:
#@title 1. 挂载 Google Drive 并解压代码与数据集（持久化软链接）
import os
from pathlib import Path
from google.colab import drive

print("=== 1. 正在挂载 Google Drive ===")
drive.mount('/content/drive')

WORKSPACE = Path("/content/gpbsf")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORKSPACE))

print("=== 2. 正在自动扫描 Google Drive 根目录压缩包 ===")
drive_root = Path("/content/drive/MyDrive")

# 智能匹配代码压缩包
code_candidates = list(drive_root.glob("*code*.zip")) + list(Path("/content").glob("*code*.zip"))
if not code_candidates:
    raise FileNotFoundError(f"在 {drive_root} 未找到包含 'code' 的 zip 文件，请检查云盘根目录。")
code_zip = code_candidates[0]
print(f"找到代码压缩包: {code_zip}")

# 智能匹配数据集压缩包
dataset_candidates = (
    list(drive_root.glob("*dataset*.zip")) +
    list(drive_root.glob("*bgspcd*.zip")) +
    list(Path("/content").glob("*dataset*.zip"))
)
if not dataset_candidates:
    raise FileNotFoundError(f"在 {drive_root} 未找到包含 'dataset' 或 'bgspcd' 的 zip 文件，请检查云盘根目录。")
dataset_zip = dataset_candidates[0]
print(f"找到数据集压缩包: {dataset_zip}")

print(f"正在解压代码到 {WORKSPACE} ...")
!unzip -q -o "{code_zip}" -d "{WORKSPACE}"

data_target = WORKSPACE / "data" / "bgspcd_v4_robust"
data_target.mkdir(parents=True, exist_ok=True)
print(f"正在解压数据集到 {data_target} ...")
!unzip -q -o "{dataset_zip}" -d "{data_target}"

# 兼容可能带有多层根目录解压的情况
manifest_file = data_target / "manifest.jsonl"
if not manifest_file.exists():
    found = list(data_target.rglob("manifest.jsonl"))
    if found:
        source_dir = found[0].parent
        !mv "{source_dir}"/* "{data_target}"/

# 核心持久化保险：将输出 runs/ 目录永久软链接到 Google 云盘
drive_runs = drive_root / "gpbsf_runs"
drive_runs.mkdir(parents=True, exist_ok=True)
local_runs = WORKSPACE / "runs"
if local_runs.exists() and not local_runs.is_symlink():
    !cp -rn "{local_runs}"/* "{drive_runs}"/ 2>/dev/null || true
    !rm -rf "{local_runs}"
!ln -sfn "{drive_runs}" "{local_runs}"
print(f"💾 输出目录已持久化软链接至 Google Drive: {drive_runs} -> {local_runs}")

print("✅ 代码与数据集准备完毕！")
!ls -lh "{data_target}/manifest.jsonl"

In [ ]:
#@title 2. GPU 与 JIT 算子健康验证
import torch
from pathlib import Path
import os

os.chdir("/content/gpbsf")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch 版本: {torch.__version__} | 计算设备: {device}")
if device == 'cuda':
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")

from experiments.classification.adapters.pointnet2 import PointNet2Adapter
adapter = PointNet2Adapter(Path('.'), device)
dummy = torch.randn(2, 2048, 3, device=device)
out = adapter.logits(dummy)
print("✅ PointNet++ JIT 向量化模型前向通过，输出张量形状:", out.shape)

In [ ]:
#@title 3. 启动 PointNet++ Seed 3409 极速训练 (125 Epochs)
import os
os.chdir("/content/gpbsf")

# 确保配置文件中 num_workers 为 0 (全量 RAM 常驻内存切片，零 IPC 延迟)
!sed -i 's/"num_workers": [0-9]*/"num_workers": 0/g' config/experiments/classification_v4_robust.json

print("=== 启动 PointNet++ 训练 | 目标种子: 3409 | 轮次: 125 ===")
# 前台实时输出，每一轮均带有系统时间戳与精确耗时(s)，自动断点续跑
!python3 -m experiments.classification.cli \
    --config config/experiments/classification_v4_robust.json train \
    --model pointnet2 \
    --seed 3409

In [ ]:
#@title 4. 测试集评估与成果自动保存至 Google Drive
import os
from pathlib import Path
from google.colab import files

os.chdir("/content/gpbsf")

print("=== 正在执行测试集评估 (Seed 3409) ===")
!python3 -m experiments.classification.cli \
    --config config/experiments/classification_v4_robust.json evaluate \
    --run-dir runs/classification_v4_robust/pointnet2/seed_3409 \
    --split test

print("=== 打包生成成果压缩包 ===")
output_tar = Path("/content/pointnet2_seed_3409_results.tar.gz")
!tar -czvf "{output_tar}" -C /content/gpbsf runs/classification_v4_robust/pointnet2/seed_3409

# 自动保存到 Google Drive 根目录
drive_target = Path("/content/drive/MyDrive/pointnet2_seed_3409_results.tar.gz")
!cp "{output_tar}" "{drive_target}"
print(f"💾 成果压缩包已自动永久保存至 Google 云盘: {drive_target}")

# 同时触发本地浏览器备用下载
try:
    files.download(str(output_tar))
except Exception:
    pass
print("🎉 全流程圆满完成！")